# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.pipeline import Pipeline

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [2]:
class FeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, df, y=None):
        return self
    
    def transform(self, df):
        df = df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour
        df['weekday'] = df['timestamp'].dt.weekday
        df = df.drop(columns=['timestamp'])
        return df

In [3]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, target_column):
        self.target_column = target_column
        self.encoder = None
        self.categorical_cols = None
        
    def fit(self, X, y=None):
        self.categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        if self.target_column in self.categorical_cols:
            self.categorical_cols.remove(self.target_column)

        self.encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
        self.encoder.fit(X[self.categorical_cols])
        return self

    def transform(self, X):
        X = X.copy()
        if self.categorical_cols:
            encoded = self.encoder.transform(X[self.categorical_cols])
            encoded_df = pd.DataFrame(encoded, columns=self.encoder.get_feature_names(self.categorical_cols), index=X.index)
            X = X.drop(columns=self.categorical_cols)
            X = pd.concat([X, encoded_df], axis=1)

        y = X[self.target_column]
        X = X.drop(columns=[self.target_column])
        return X, y

In [4]:
class TrainValidationTest:
    def __init__(self, test_size=0.2, valid_size=0.25, random_state=21, min_samples_per_class=2, verbose=True):
        self.test_size = test_size
        self.valid_size = valid_size
        self.random_state = random_state
        self.min_samples_per_class = min_samples_per_class
        self.verbose = verbose

    def _can_stratify(self, y):
        class_counts = pd.Series(y).value_counts()
        can = (class_counts >= self.min_samples_per_class).all()
        if self.verbose:
            if can:
                print(f"Stratified splitting enabled (all classes have >= {self.min_samples_per_class} samples).")
            else:
                problem_classes = class_counts[class_counts < self.min_samples_per_class]
                print(f"Stratified splitting disabled: some classes too small:\n{problem_classes}")
        return can

    def split(self, X, y):
        stratify_flag = self._can_stratify(y)

        X_train_valid, X_test, y_train_valid, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y if stratify_flag else None
        )

        stratify_flag_2 = self._can_stratify(y_train_valid)

        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_valid, y_train_valid,
            test_size=self.valid_size,
            random_state=self.random_state,
            stratify=y_train_valid if stratify_flag_2 else None
        )

        return X_train, X_valid, X_test, y_train, y_valid, y_test
    

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.772727
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.801484
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.855288
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [5]:
class ModelSelection:
    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models_info = []
    
    def choose(self, X_train, y_train, X_valid, y_valid):
        valid_scores = []
        
        for idx, grid in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            print(f"\nEstimator: {model_name}")

            with tqdm(total=len(grid.param_grid), desc=f"Grid search for {model_name}") as pbar:
                for params in grid.param_grid:
                    grid.set_params(param_grid=[params])
                    grid.fit(X_train, y_train)
                    pbar.update(1)

            grid.set_params(param_grid=grid.param_grid)
            grid.fit(X_train, y_train)
            
            best_params = grid.best_params_
            train_score = grid.best_score_
            best_model = grid.best_estimator_

            valid_pred = best_model.predict(X_valid)
            valid_acc = accuracy_score(y_valid, valid_pred)
            
            self.best_models_info.append({
                'model': model_name,
                'params': best_params,
                'train_score': train_score,
                'valid_score': valid_acc
            })

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_acc:.3f}\n")
            
            valid_scores.append(valid_acc)
        
        best_idx = valid_scores.index(max(valid_scores))
        best_model_name = self.grid_dict[best_idx]
        print(f"Classifier with best validation set accuracy: {best_model_name}")
        
        return best_model_name

    def best_results(self):
        return pd.DataFrame(self.best_models_info)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [14]:
class Finalize:
    def __init__(self, estimator):
        self.estimator = estimator
        
    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {acc:.5f}")
        return acc
    
    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model was successfully saved to {path}")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [7]:
df = pd.read_csv("checker_submits.csv")

In [8]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('weekday'))
])

X_processed, y = preprocessing.fit_transform(df)

In [9]:
splitter = TrainValidationTest()
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.split(X_processed, y)

print('X_train shape: ', X_train.shape)
print('y_train shape: ', y_train.shape)
print('X_valid shape: ', X_valid.shape)
print('y_valid shape: ', y_valid.shape)
print('X_test shape : ', X_test.shape)
print('y_test shape : ', y_test.shape)

Stratified splitting enabled (all classes have >= 2 samples).
Stratified splitting enabled (all classes have >= 2 samples).
X_train shape:  (1011, 43)
y_train shape:  (1011,)
X_valid shape:  (337, 43)
y_valid shape:  (337,)
X_test shape :  (338, 43)
y_test shape :  (338,)


In [10]:
svm_params = [{
    'kernel': ('linear', 'rbf', 'sigmoid'),
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ('balanced', None),
    'random_state': [21],
    'probability': [True]
}]

tree_params = [{
    'criterion': ['gini', 'entropy'],
    'max_depth': [5, 10, 15, 20, 21, 22],
    'class_weight': ('balanced', None),
    'random_state': [21]
}]

rf_params = [{
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 15, 20, 22],
    'n_estimators': [10, 20, 50],
    'class_weight': ('balanced', None),
    'random_state': [21]
}]

jobs = -1

gs_svm = GridSearchCV(SVC(), param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs)
gs_tree = GridSearchCV(DecisionTreeClassifier(), param_grid=tree_params, scoring='accuracy', cv=2, n_jobs=jobs)
gs_rf = GridSearchCV(RandomForestClassifier(), param_grid=rf_params, scoring='accuracy', cv=2, n_jobs=jobs)


In [11]:
grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: 'SVM', 1: 'Decision Tree', 2: 'Random Forest'}

selector = ModelSelection(grids, grid_dict)
best_model = selector.choose(X_train, y_train, X_valid, y_valid)
results = selector.best_results()
print(results)


Estimator: SVM


Grid search for SVM: 100%|██████████| 1/1 [01:48<00:00, 108.46s/it]


Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.752
Validation set accuracy score for best params: 0.855


Estimator: Decision Tree


Grid search for Decision Tree: 100%|██████████| 1/1 [00:00<00:00,  5.22it/s]


Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 20, 'random_state': 21}
Best training accuracy: 0.802
Validation set accuracy score for best params: 0.869


Estimator: Random Forest


Grid search for Random Forest: 100%|██████████| 1/1 [00:01<00:00,  1.79s/it]


Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.854
Validation set accuracy score for best params: 0.878

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': None, 'gamma': 'auto...   
1  Decision Tree  {'class_weight': None, 'criterion': 'gini', 'm...   
2  Random Forest  {'class_weight': 'balanced', 'criterion': 'ent...   

   train_score  valid_score  
0     0.751714     0.854599  
1     0.802201     0.869436  
2     0.853630     0.878338  


In [12]:
model_name_to_idx = {v: k for k, v in grid_dict.items()}
best_grid = grids[model_name_to_idx[best_model]]
best_estimator = best_grid.best_estimator_

In [15]:
final = Finalize(best_estimator)
final.final_score(X_train, y_train, X_test, y_test)
final.save_model('best_model.joblib')

Accuracy of the final model is 0.90828
Model was successfully saved to best_model.joblib
